<div style="background:#0B1F4B;padding:22px 26px;border-radius:12px;border:3px solid #C9A227;font-family:'Times New Roman',serif">
<img src="https://raw.githubusercontent.com/drssnaiml1-cell/26UAMPC204-Artificial-Intelligence/main/assets/logo.png" alt="Sanjivani University" width="92" style="float:right;background:#FFFFFF;border:2px solid #C9A227;border-radius:8px;padding:3px;margin-left:14px">
<div style="color:#C9A227;font-size:14px;letter-spacing:2px;font-weight:bold">26UAMPC204 · ARTIFICIAL INTELLIGENCE · ASSIGNMENT 4 · UNIT IV · CO204.4</div>
<div style="color:#FFFFFF;font-size:30px;font-weight:bold;margin-top:6px">Reasoning When Nothing Is Certain</div>
<div style="color:#F6EBC8;font-size:15px;margin-top:8px">Due: End of Week 12 · Name: ____________ · Roll No: ________</div>
<div style="clear:both"></div></div>

## 🧠 Part A — Think out of the box

Answer each in 250–400 words in the markdown cells below.

1. **[L5]** A COVID-style rapid test is advertised as '95% accurate'. Explain to your family, without formulas, why a positive result might still be more likely wrong than right. When does it become reliable?
2. **[L6]** Design a STRIPS model for 'getting ready for college in the morning'. Where does the classical-planning assumption fail in real life, and how would you repair the plan on the fly?
3. **[L4]** Is 'tall' better modelled by probability or fuzzy logic? Construct a situation where using the wrong one leads to a bad decision.
4. **[L5]** Bayesian networks need a causal structure. What goes wrong if an expert draws an arrow in the wrong direction? Propose a test to detect it from data.

**Answer A4.1:** _write here_

**Answer A4.2:** _write here_

**Answer A4.3:** _write here_

**Answer A4.4:** _write here_

## 💻 Part B — Open-ended coding

The starter code runs as-is and prints a baseline. Complete every `TODO`.

### B4.1 · Open-ended: build a Bayesian network for a decision you care about

<div style="border-left:6px solid #1F3A7A;background:#EEF2F8;padding:12px 18px;border-radius:6px;font-family:'Times New Roman',serif;color:#1A202C">
<b style="color:#1F3A7A">Task</b><br>Choose a diagnosis or decision (laptop won't boot, will it rain at college, should I take the bus). Build a network with ≥ 5 nodes, justify every CPT, and answer three queries with enumeration.</div>

In [1]:
import itertools
# Starter: Rain -> WetRoad <- Sprinkler ; WetRoad -> Slip
P_R, P_S = 0.3, 0.2
P_W = {(True, True): 0.99, (True, False): 0.9, (False, True): 0.8, (False, False): 0.05}
P_SL = {True: 0.2, False: 0.01}
def joint(r, s, w, sl):
    p = (P_R if r else 1 - P_R) * (P_S if s else 1 - P_S)
    p *= P_W[(r, s)] if w else 1 - P_W[(r, s)]
    return p * (P_SL[w] if sl else 1 - P_SL[w])
def query(var_index, evidence):
    num = {True: 0.0, False: 0.0}
    for vals in itertools.product((True, False), repeat=4):
        if all(vals[i] == v for i, v in evidence.items()): num[vals[var_index]] += joint(*vals)
    z = sum(num.values()); return {k: round(v / z, 4) for k, v in num.items()}
print("P(Rain | Slip) =", query(0, {3: True}))
# TODO: design YOUR network (>= 5 nodes), justify CPTs from data or experts,
#       answer three queries and run a sensitivity analysis on one CPT entry.

P(Rain | Slip) = {True: 0.6222, False: 0.3778}


In [2]:
# Your solution for B4.1


### B4.2 · Open-ended: fuzzy controller you can feel

<div style="border-left:6px solid #1F3A7A;background:#EEF2F8;padding:12px 18px;border-radius:6px;font-family:'Times New Roman',serif;color:#1A202C">
<b style="color:#1F3A7A">Task</b><br>Design a fuzzy controller for something physical (fan speed, phone brightness, shower temperature). Plot membership functions and the control surface, then argue whether a simple formula would do better.</div>

In [3]:
import numpy as np
def tri(x, a, b, c): return np.maximum(np.minimum((x - a) / (b - a + 1e-9), (c - x) / (c - b + 1e-9)), 0)
def fan_speed(temp):
    cold, warm, hot = tri(temp, 0, 10, 22), tri(temp, 15, 25, 35), tri(temp, 28, 40, 52)
    z = np.linspace(0, 100, 501)
    low, med, high = tri(z, -1, 0, 50), tri(z, 20, 50, 80), tri(z, 50, 100, 101)
    agg = np.maximum.reduce([np.minimum(cold, low), np.minimum(warm, med), np.minimum(hot, high)])
    return float((agg * z).sum() / (agg.sum() + 1e-9))      # centroid defuzzification
for t in (12, 24, 31, 38): print(f"{t} °C -> fan speed {fan_speed(t):.1f}%")
# TODO: add a second input (humidity), write >= 6 rules, plot the 3-D control surface,
#       and compare with a hand-written linear formula.

12 °C -> fan speed 17.0%
24 °C -> fan speed 50.0%
31 °C -> fan speed 59.0%
38 °C -> fan speed 83.0%


In [4]:
# Your solution for B4.2


### B4.3 · Open-ended: planning with surprises

<div style="border-left:6px solid #1F3A7A;background:#EEF2F8;padding:12px 18px;border-radius:6px;font-family:'Times New Roman',serif;color:#1A202C">
<b style="color:#1F3A7A">Task</b><br>Implement a forward STRIPS planner for a delivery robot. Then make one action randomly fail and add execution monitoring with re-planning. Measure how often goals are still reached.</div>

In [5]:
from collections import deque
ACTIONS = {  # name: (preconditions, add, delete)
    "move(A,B)": ({"at(A)"}, {"at(B)"}, {"at(A)"}), "move(B,A)": ({"at(B)"}, {"at(A)"}, {"at(B)"}),
    "move(B,C)": ({"at(B)"}, {"at(C)"}, {"at(B)"}), "move(C,B)": ({"at(C)"}, {"at(B)"}, {"at(C)"}),
    "pick(A)": ({"at(A)", "parcel(A)", "free"}, {"holding"}, {"parcel(A)", "free"}),
    "drop(C)": ({"at(C)", "holding"}, {"parcel(C)", "free"}, {"holding"}),
}
def plan(state, goal):
    q, seen = deque([(frozenset(state), [])]), set()
    while q:
        s, p = q.popleft()
        if goal <= s: return p
        if s in seen: continue
        seen.add(s)
        for a, (pre, add, dele) in ACTIONS.items():
            if pre <= s: q.append(((s - dele) | add, p + [a]))
print(plan({"at(B)", "parcel(A)", "free"}, {"parcel(C)"}))
# TODO: make 'move' fail with probability 0.2, monitor the state after each step,
#       re-plan on failure, and report the success rate and extra steps over 1000 runs.

['move(B,A)', 'pick(A)', 'move(A,B)', 'move(B,C)', 'drop(C)']


In [6]:
# Your solution for B4.3


## 📋 Reflection

What surprised you? What would you do with one more week? How did you use AI tools, if at all?